# How do I evolve a RAG application into an agent memory system?

This notebook starts with a working retrieval-augmented generation (RAG) application and evolves it one capability at a time into a scoped, durable agent memory system on Oracle AI Database.

The implementation uses Python and SQL so every persistence, retrieval, correction, and deletion decision remains visible.


## What You Will Build

By the end of the notebook, you will have:

- a small RAG knowledge base backed by Oracle AI Vector Search
- a reproducible example showing why RAG alone cannot remember a user across sessions
- durable memories scoped by tenant, user, agent, and optional thread
- explicit memory promotion instead of saving every message indiscriminately
- one context assembly path that keeps retrieved evidence and remembered experience separate
- correction, supersession, forgetting, isolation, and before/after evaluation checks

### The central distinction

| Layer | Contains | How it changes |
|---|---|---|
| RAG knowledge | Product documentation and support procedures | Ingested or refreshed by the application |
| Agent memory | User preferences, prior issues, and useful outcomes | Created through interaction and policy |
| System of record | Tickets, accounts, and authoritative business state | Changed only through approved business workflows |


## Architecture at a Glance

```text
User question
      |
      v
[Retrieve knowledge + scoped memory]
      |
      v
[Assemble separately labeled context]
      |
      v
[Generate a grounded answer]
      |
      v
Safe, durable memory candidate?
      |
      +---- Yes ----> Store or update memory in Oracle AI Database
      |
      +---- No -----> End the turn without creating memory
```

RAG remains the evidence path. Memory becomes a second, explicitly scoped
context path; it does not turn remembered text into authoritative truth.

**Embedding note:** The default `local` provider keeps the tutorial self-contained
with deterministic feature hashing. Set `RAG_MEMORY_EMBEDDING_PROVIDER=openai` to
run the same interfaces with semantic embeddings. The semantic option sends the
synthetic tutorial documents and memories to the configured embedding API.

## Notebook Structure

1. Prepare the local runtime and validate Oracle AI Database.
2. Create a fresh schema and seed a compact support knowledge base.
3. Build and test a stateless RAG baseline.
4. Prove that RAG cannot recall interaction-derived facts across sessions.
5. Define a memory contract and add a new SQL-backed memory layer.
6. Retrieve scoped memories and assemble them beside RAG evidence.
7. Demonstrate user isolation, correction, supersession, and forgetting.
8. Run deterministic acceptance checks and optionally remove this run's data.


## Prerequisites

- Python 3.10+ and a Jupyter kernel in VS Code or JupyterLab.
- Oracle AI Database access.
- A local application user with table-create privileges in `FREEPDB1`.
- `oracledb`, `openai`, `python-dotenv`, and `pandas` packages installed.
- A database user that can create tables and use Oracle AI Vector Search.

## Local Oracle Database Setup

This notebook uses a local Oracle Database Free container running in Podman and publishing database port `1521` to the host.

If the container does not already exist, create it with:

```bash
podman pull gvenzl/oracle-free:23.26.2
podman volume create oracle-free-data

podman run -d \
  --name oracle-free \
  -p 1521:1521 \
  -e ORACLE_PASSWORD="<admin-password>" \
  -e APP_USER="<app-user>" \
  -e APP_USER_PASSWORD="<password>" \
  -v oracle-free-data:/opt/oracle/oradata \
  gvenzl/oracle-free:23.26.2

podman logs -f oracle-free
```

Then configure the active connection in `.env`:

```env
ORACLE_USER=<application user>
ORACLE_PASSWORD=<password>
ORACLE_DSN=localhost:1521/FREEPDB1
LOCAL_EMBEDDING_DIMENSION=384
RAG_MEMORY_EMBEDDING_PROVIDER=local
OPENAI_API_KEY=
RUN_OPENAI_ANSWERS=0
OPENAI_EMBEDDING_MODEL=text-embedding-3-small
OPENAI_EMBEDDING_DIMENSION=384
OPENAI_MODEL=gpt-4.1-mini
```

The executed notebook uses the local Podman `oracle-free` container.

## Step 1 - Prepare the Local Runtime


This step verifies that the required Python packages are available before the notebook does any database work. The following cells then locate `.env`, validate the embedding and answer-provider settings, create an Oracle connection pool, and run a temporary vector-table preflight so configuration problems fail early.

In [1]:
import sys
from importlib.metadata import PackageNotFoundError, version


required_packages = ["oracledb", "openai", "python-dotenv", "pandas"]

missing = []
for package_name in required_packages:
    try:
        installed_version = version(package_name)
        print(f"{package_name}: {installed_version}")
    except PackageNotFoundError:
        missing.append(package_name)

if missing:
    raise RuntimeError(
        "Missing packages: " + ", ".join(missing) +
        ". Install them with: %pip install -U oracledb openai python-dotenv pandas"
    )

print("Python runtime:", sys.version.split()[0])
print("Runtime dependencies: READY")


oracledb: 4.0.2
openai: 2.44.0
python-dotenv: 1.2.3
pandas: 3.0.5
Python runtime: 3.13.14
Runtime dependencies: READY


### Load configuration


In [2]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv


def find_project_directory(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / ".env").exists():
            return candidate
    raise FileNotFoundError(
        "No .env file was found in the notebook folder or its parents."
    )


configured_project_directory = os.getenv("RAG_MEMORY_PROJECT_DIR")
PROJECT_DIR = (
    Path(configured_project_directory).expanduser().resolve()
    if configured_project_directory
    else find_project_directory(Path.cwd())
)
if not (PROJECT_DIR / ".env").exists():
    raise FileNotFoundError(f"No .env file found in configured project directory: {PROJECT_DIR}")
load_dotenv(PROJECT_DIR / ".env", override=False)

CONFIG = {
    "DB_USER": os.getenv("DB_USER") or os.getenv("ORACLE_USER"),
    "DB_PASSWORD": os.getenv("DB_PASSWORD") or os.getenv("ORACLE_PASSWORD"),
    "DB_DSN": (
        os.getenv("DB_CONNECT_STRING")
        or os.getenv("DB_DSN")
        or os.getenv("ORACLE_DSN")
    ),
    "OPENAI_API_KEY": os.getenv("OPENAI_API_KEY"),
    "EMBEDDING_PROVIDER": os.getenv("RAG_MEMORY_EMBEDDING_PROVIDER", "local").lower(),
    "LOCAL_EMBEDDING_DIM": int(os.getenv("LOCAL_EMBEDDING_DIMENSION") or "384"),
    "OPENAI_EMBEDDING_MODEL": os.getenv("OPENAI_EMBEDDING_MODEL") or "text-embedding-3-small",
    "OPENAI_EMBEDDING_DIM": int(os.getenv("OPENAI_EMBEDDING_DIMENSION") or "384"),
    "OPENAI_MODEL": os.getenv("OPENAI_MODEL") or "gpt-4.1-mini",
    "RUN_OPENAI_ANSWERS": os.getenv("RUN_OPENAI_ANSWERS", "1") == "1",
}

required = ["DB_USER", "DB_PASSWORD", "DB_DSN"]
missing = [key for key in required if not CONFIG[key]]
if missing:
    raise RuntimeError("Missing required configuration values: " + ", ".join(missing))

if CONFIG["EMBEDDING_PROVIDER"] not in {"local", "openai"}:
    raise ValueError("RAG_MEMORY_EMBEDDING_PROVIDER must be local or openai.")
CONFIG["EMBEDDING_DIM"] = (
    CONFIG["OPENAI_EMBEDDING_DIM"]
    if CONFIG["EMBEDDING_PROVIDER"] == "openai"
    else CONFIG["LOCAL_EMBEDDING_DIM"]
)
if not 32 <= CONFIG["EMBEDDING_DIM"] <= 4096:
    raise ValueError("The configured embedding dimension must be between 32 and 4096.")
if CONFIG["EMBEDDING_PROVIDER"] == "openai" and not CONFIG["OPENAI_API_KEY"]:
    raise RuntimeError("OPENAI_API_KEY is required for the OpenAI embedding provider.")

LIVE_ANSWERS = bool(CONFIG["OPENAI_API_KEY"]) and CONFIG["RUN_OPENAI_ANSWERS"]
print("Database configuration: READY")
print("Embedding provider:", CONFIG["EMBEDDING_PROVIDER"])
print("Answer generation:", "OPENAI" if LIVE_ANSWERS else "LOCAL GROUNDED FALLBACK")


Database configuration: READY
Embedding provider: local
Answer generation: LOCAL GROUNDED FALLBACK


### Connect and run a database preflight


In [ ]:
import hashlib
import json
import math
from uuid import uuid4

import oracledb
from openai import OpenAI


oracledb.defaults.program = "devrel-developerhub-rag-agent-memory-notebook"
pool = oracledb.create_pool(
    user=CONFIG["DB_USER"],
    password=CONFIG["DB_PASSWORD"],
    dsn=CONFIG["DB_DSN"],
    min=1,
    max=4,
    increment=1,
)

openai_client = (
    OpenAI(api_key=CONFIG["OPENAI_API_KEY"])
    if CONFIG["OPENAI_API_KEY"]
    else None
)

def embed_texts(texts: list[str]) -> list[list[float]]:
    cleaned = [str(text).lower().replace("\\n", " ").strip() for text in texts]
    if not cleaned or any(not item for item in cleaned):
        raise ValueError("embedding input must contain non-empty text")

    if CONFIG["EMBEDDING_PROVIDER"] == "openai":
        response = openai_client.embeddings.create(
            model=CONFIG["OPENAI_EMBEDDING_MODEL"],
            input=cleaned,
            dimensions=CONFIG["EMBEDDING_DIM"],
            encoding_format="float",
        )
        vectors = [
            item.embedding
            for item in sorted(response.data, key=lambda result: result.index)
        ]
        if any(len(vector) != CONFIG["EMBEDDING_DIM"] for vector in vectors):
            raise RuntimeError("Embedding response dimension did not match configuration.")
        return vectors

    vectors = []
    for item in cleaned:
        tokens = re.findall(r"[a-z0-9]+", item)
        features = [(token, 2.0) for token in tokens]
        features.extend(
            (f"{left}_{right}", 1.5)
            for left, right in zip(tokens, tokens[1:])
        )
        for token in tokens:
            padded = f"^{token}$"
            features.extend(
                (padded[index:index + 3], 0.5)
                for index in range(max(0, len(padded) - 2))
            )

        vector = [0.0] * CONFIG["EMBEDDING_DIM"]
        for feature, weight in features:
            digest = hashlib.blake2b(feature.encode("utf-8"), digest_size=8).digest()
            bucket = int.from_bytes(digest[:4], "big") % CONFIG["EMBEDDING_DIM"]
            sign = 1.0 if digest[4] % 2 == 0 else -1.0
            vector[bucket] += sign * weight
        norm = math.sqrt(sum(value * value for value in vector)) or 1.0
        vectors.append([value / norm for value in vector])
    return vectors


probe_vector = embed_texts(["Oracle vector preflight"])[0]
check_table = f"RAG_MEM_CHECK_{uuid4().hex[:8].upper()}"

with pool.acquire() as connection:
    try:
        with connection.cursor() as cursor:
            cursor.execute("SELECT 1 FROM dual")
            assert cursor.fetchone()[0] == 1
            cursor.execute(
                f"CREATE TABLE {check_table} (id NUMBER PRIMARY KEY, note VARCHAR2(80))"
            )
            cursor.execute(
                f"INSERT INTO {check_table} (id, note) VALUES (:id, :note)",
                id=1,
                note="preflight",
            )
            cursor.execute(f"SELECT COUNT(*) FROM {check_table}")
            assert cursor.fetchone()[0] == 1
            cursor.execute(
                f"""SELECT VECTOR_DISTANCE(
                        TO_VECTOR(:left_vector, {CONFIG['EMBEDDING_DIM']}, FLOAT32),
                        TO_VECTOR(:right_vector, {CONFIG['EMBEDDING_DIM']}, FLOAT32),
                        COSINE
                    ) FROM dual""",
                left_vector=json.dumps(probe_vector),
                right_vector=json.dumps(probe_vector),
            )
            assert abs(float(cursor.fetchone()[0])) < 1e-6
        connection.commit()
    finally:
        with connection.cursor() as cursor:
            try:
                cursor.execute(f"DROP TABLE {check_table} PURGE")
            except oracledb.Error as exc:
                if exc.args[0].code != 942:
                    raise
        connection.commit()

print("Oracle AI Database connection: READY")
print("Create/read/drop privileges: READY")
print("Embedding provider validation: READY")
print("Oracle vector operations: READY")


Oracle AI Database connection: READY
Create/read/drop privileges: READY
Embedding provider validation: READY
Oracle vector operations: READY


## Step 2 - Create a Schema


The following cell creates four tutorial tables when they do not already exist. Every row is isolated by a random `run_id`, vector columns use the configured embedding dimension, and an active-memory index prevents two current values for the same scoped memory key.

| Table | Responsibility |
|---|---|
| `RAG_EVOLVE_KNOWLEDGE` | Stable document chunks used by RAG |
| `RAG_EVOLVE_THREADS` | User/agent conversation identity |
| `RAG_EVOLVE_TURNS` | Append-only conversational evidence |
| `RAG_EVOLVE_MEMORIES` | Promoted durable memories and their lifecycle |

In [4]:
from datetime import datetime, timezone


RUN_ID = uuid4().hex[:20]
VECTOR_DIM = CONFIG["EMBEDDING_DIM"]


def create_table_if_missing(connection, table_name, ddl):
    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM user_tables WHERE table_name = :table_name",
            table_name=table_name.upper(),
        )
        exists = cursor.fetchone()[0] == 1
        if not exists:
            cursor.execute(ddl)
    connection.commit()


table_definitions = {
    "RAG_EVOLVE_KNOWLEDGE": f"""
        CREATE TABLE RAG_EVOLVE_KNOWLEDGE (
            run_id       VARCHAR2(32) NOT NULL,
            chunk_id     VARCHAR2(64) NOT NULL,
            source_name  VARCHAR2(200) NOT NULL,
            content      CLOB NOT NULL,
            embedding    VECTOR({VECTOR_DIM}, FLOAT32) NOT NULL,
            created_at   TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            CONSTRAINT pk_rag_evolve_knowledge PRIMARY KEY (run_id, chunk_id)
        )
    """,
    "RAG_EVOLVE_THREADS": """
        CREATE TABLE RAG_EVOLVE_THREADS (
            run_id       VARCHAR2(32) NOT NULL,
            tenant_id    VARCHAR2(100) NOT NULL,
            user_id      VARCHAR2(200) NOT NULL,
            agent_id     VARCHAR2(100) NOT NULL,
            thread_id    VARCHAR2(100) NOT NULL,
            created_at   TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            updated_at   TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            CONSTRAINT pk_rag_evolve_threads
                PRIMARY KEY (run_id, tenant_id, user_id, agent_id, thread_id)
        )
    """,
    "RAG_EVOLVE_TURNS": """
        CREATE TABLE RAG_EVOLVE_TURNS (
            turn_id      VARCHAR2(32) PRIMARY KEY,
            run_id       VARCHAR2(32) NOT NULL,
            tenant_id    VARCHAR2(100) NOT NULL,
            user_id      VARCHAR2(200) NOT NULL,
            agent_id     VARCHAR2(100) NOT NULL,
            thread_id    VARCHAR2(100) NOT NULL,
            role         VARCHAR2(20) NOT NULL,
            content      CLOB NOT NULL,
            created_at   TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            CONSTRAINT ck_rag_evolve_turn_role
                CHECK (role IN ('user', 'assistant', 'tool'))
        )
    """,
    "RAG_EVOLVE_MEMORIES": f"""
        CREATE TABLE RAG_EVOLVE_MEMORIES (
            memory_id            VARCHAR2(32) PRIMARY KEY,
            run_id               VARCHAR2(32) NOT NULL,
            tenant_id            VARCHAR2(100) NOT NULL,
            user_id              VARCHAR2(200) NOT NULL,
            agent_id             VARCHAR2(100) NOT NULL,
            thread_id            VARCHAR2(100),
            memory_key           VARCHAR2(160) NOT NULL,
            memory_type          VARCHAR2(30) NOT NULL,
            content              CLOB NOT NULL,
            embedding            VECTOR({VECTOR_DIM}, FLOAT32) NOT NULL,
            status               VARCHAR2(20) DEFAULT 'active' NOT NULL,
            supersedes_memory_id VARCHAR2(32),
            source_turn_id       VARCHAR2(32),
            importance           NUMBER(2) DEFAULT 3 NOT NULL,
            created_at           TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            updated_at           TIMESTAMP WITH TIME ZONE DEFAULT SYSTIMESTAMP NOT NULL,
            CONSTRAINT ck_rag_evolve_memory_type
                CHECK (memory_type IN ('preference', 'fact', 'episode')),
            CONSTRAINT ck_rag_evolve_memory_status
                CHECK (status IN ('active', 'superseded', 'deleted')),
            CONSTRAINT ck_rag_evolve_importance
                CHECK (importance BETWEEN 1 AND 5)
        )
    """,
}

with pool.acquire() as connection:
    for table_name, ddl in table_definitions.items():
        create_table_if_missing(connection, table_name, ddl)

    with connection.cursor() as cursor:
        cursor.execute(
            "SELECT COUNT(*) FROM user_indexes WHERE index_name = 'UQ_RAG_EVOLVE_ACTIVE_MEMORY'"
        )
        if cursor.fetchone()[0] == 0:
            cursor.execute(
                """
                CREATE UNIQUE INDEX UQ_RAG_EVOLVE_ACTIVE_MEMORY
                    ON RAG_EVOLVE_MEMORIES (
                        CASE WHEN status = 'active' THEN run_id END,
                        CASE WHEN status = 'active' THEN tenant_id END,
                        CASE WHEN status = 'active' THEN user_id END,
                        CASE WHEN status = 'active' THEN agent_id END,
                        CASE WHEN status = 'active' THEN NVL(thread_id, '__GLOBAL__') END,
                        CASE WHEN status = 'active' THEN memory_key END
                    )
                """
            )
    connection.commit()

print("Tutorial tables: READY")
print("Run isolation: READY")


Tutorial tables: READY
Run isolation: READY


## Step 3 - Seed a Compact RAG Knowledge Base


The scenario is an `Aurora Sync` support assistant.

The following cell defines four synthetic product and support documents, embeds them with the configured provider, and inserts the vectors into `RAG_EVOLVE_KNOWLEDGE` for this run. These documents intentionally exclude the user's current issue and communication preference.

In [5]:
KNOWLEDGE_CHUNKS = [
    {
        "chunk_id": "sync-204-resolution",
        "source_name": "aurora-sync-troubleshooting.md",
        "content": (
            "Error SYNC-204 means the desktop client lost its refresh token. "
            "Resolve it by signing out, removing the cached session, signing in again, "
            "and confirming that a manual sync completes. Escalate if the error returns."
        ),
    },
    {
        "chunk_id": "support-notifications",
        "source_name": "support-notifications.md",
        "content": (
            "Support updates can be delivered by email or Slack when the customer "
            "has selected a channel. If no preference is recorded, ask the customer "
            "instead of choosing a channel."
        ),
    },
    {
        "chunk_id": "sync-diagnostics",
        "source_name": "aurora-sync-diagnostics.md",
        "content": (
            "Before escalating a synchronization incident, collect the client version, "
            "the most recent error code, the timestamp of the failed sync, and whether "
            "a manual retry succeeded."
        ),
    },
    {
        "chunk_id": "account-security",
        "source_name": "account-security.md",
        "content": (
            "Never store passwords, access tokens, or authentication secrets in support "
            "notes or assistant memory. Use approved credential and secret-management systems."
        ),
    },
]

insert_knowledge_sql = f"""
    INSERT INTO RAG_EVOLVE_KNOWLEDGE
        (run_id, chunk_id, source_name, content, embedding)
    VALUES
        (:run_id, :chunk_id, :source_name, :content,
         TO_VECTOR(:embedding_json, {VECTOR_DIM}, FLOAT32))
"""

knowledge_vectors = embed_texts([chunk["content"] for chunk in KNOWLEDGE_CHUNKS])
with pool.acquire() as connection:
    with connection.cursor() as cursor:
        for chunk, vector in zip(KNOWLEDGE_CHUNKS, knowledge_vectors):
            cursor.execute(
                insert_knowledge_sql,
                run_id=RUN_ID,
                embedding_json=json.dumps(vector),
                **chunk,
            )
    connection.commit()

print(f"Knowledge chunks loaded: {len(KNOWLEDGE_CHUNKS)}")


Knowledge chunks loaded: 4


## Step 4 - Build the Stateless RAG Baseline


This step implements the document-retrieval path used by the original RAG application. It defines `retrieve_knowledge`, converts the question into a vector, uses Oracle `VECTOR_DISTANCE` to rank chunks, and applies the same documented distance threshold to both answerable and unsupported questions.

In [6]:
import pandas as pd


def retrieve_knowledge(
    query: str,
    limit: int = 3,
    max_distance: float | None = None,
) -> list[dict]:
    if not query.strip():
        raise ValueError("query must not be empty")
    limit = max(1, min(int(limit), 10))
    query_vector = embed_texts([query])[0]
    sql = f"""
        SELECT chunk_id,
               source_name,
               DBMS_LOB.SUBSTR(content, 4000, 1) AS content,
               VECTOR_DISTANCE(
                   embedding,
                   TO_VECTOR(:query_vector, {VECTOR_DIM}, FLOAT32),
                   COSINE
               ) AS distance
          FROM RAG_EVOLVE_KNOWLEDGE
         WHERE run_id = :run_id
         ORDER BY distance
         FETCH FIRST {limit} ROWS ONLY
    """
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            rows = cursor.execute(
                sql,
                query_vector=json.dumps(query_vector),
                run_id=RUN_ID,
            ).fetchall()
    results = [
        {
            "chunk_id": row[0],
            "source_name": row[1],
            "content": row[2],
            "distance": float(row[3]),
        }
        for row in rows
    ]
    if max_distance is not None:
        results = [item for item in results if item["distance"] <= max_distance]
    return results


# Calibrate this on a held-out evaluation set before production use. The
# tutorial applies the same gate to supported and unsupported questions.
knowledge_distance_threshold = (
    0.65 if CONFIG["EMBEDDING_PROVIDER"] == "openai" else 0.70
)
baseline_question = "How should I resolve error SYNC-204?"
baseline_documents = retrieve_knowledge(
    baseline_question,
    max_distance=knowledge_distance_threshold,
)
assert baseline_documents
assert baseline_documents[0]["chunk_id"] == "sync-204-resolution"
pd.DataFrame(baseline_documents)[["chunk_id", "source_name", "distance"]]


,chunk_id,source_name,distance
0,sync-204-resolution,aurora-sync-troubleshooting.md,0.675541


### Optional semantic paraphrase check


The cell asks the same support question with different wording and inspects the retrieved chunk. It is an actual semantic-retrieval assertion only when the OpenAI embedding provider is enabled; the deterministic local provider reports the check as not run instead of claiming semantic equivalence.

In [7]:
paraphrase_question = "Why did the desktop client stop refreshing its session?"
paraphrase_documents = retrieve_knowledge(
    paraphrase_question,
    max_distance=knowledge_distance_threshold,
)
semantic_paraphrase_pass = (
    CONFIG["EMBEDDING_PROVIDER"] != "openai"
    or paraphrase_documents[0]["chunk_id"] == "sync-204-resolution"
)

if CONFIG["EMBEDDING_PROVIDER"] == "openai":
    assert semantic_paraphrase_pass
    print("Semantic paraphrase retrieval: PASS")
else:
    print(
        "Semantic paraphrase check: NOT RUN with the local feature-hashing provider. "
        "Set RAG_MEMORY_EMBEDDING_PROVIDER=openai to enable it."
    )
pd.DataFrame(
    paraphrase_documents,
    columns=["chunk_id", "source_name", "distance"],
)


Semantic paraphrase check: NOT RUN with the local feature-hashing provider. Set RAG_MEMORY_EMBEDDING_PROVIDER=openai to enable it.


,chunk_id,source_name,distance
0,sync-diagnostics,aurora-sync-diagnostics.md,0.685816
1,sync-204-resolution,aurora-sync-troubleshooting.md,0.690011


In [8]:
from openai import OpenAI


if LIVE_ANSWERS and openai_client is None:
    openai_client = OpenAI(api_key=CONFIG["OPENAI_API_KEY"])


def generate_answer(question: str, context: str) -> str:
    instructions = (
        "You are a support assistant. Answer only from the supplied context. "
        "Keep document evidence and user memory conceptually separate. "
        "If the context does not establish a user-specific fact, say that it is unknown."
    )
    prompt = f"CONTEXT\n{context}\n\nQUESTION\n{question}"
    if not LIVE_ANSWERS:
        memory_facts = re.findall(
            r"\[MEMORY [^\]]+\]\n([^\n]+)", context
        )
        document_facts = re.findall(
            r"\[DOCUMENT [^\]]+\]\n([^\n]+)", context
        )
        if not memory_facts and not document_facts:
            return "I do not have enough retrieved evidence to answer that question."

        sections = []
        if memory_facts:
            sections.append("Remembered context: " + " ".join(memory_facts))
        if document_facts:
            sections.append("Retrieved knowledge: " + " ".join(document_facts[:2]))
        return "\n\n".join(sections)
    response = openai_client.responses.create(
        model=CONFIG["OPENAI_MODEL"],
        instructions=instructions,
        input=prompt,
        store=False,
    )
    return response.output_text.strip()


def format_knowledge_context(documents: list[dict]) -> str:
    return "\n\n".join(
        f"[DOCUMENT {item['source_name']}#{item['chunk_id']}]\n{item['content']}"
        for item in documents
    )


baseline_context = format_knowledge_context(baseline_documents)
baseline_answer = generate_answer(baseline_question, baseline_context)
print(baseline_answer)


Retrieved knowledge: Error SYNC-204 means the desktop client lost its refresh token. Resolve it by signing out, removing the cached session, signing in again, and confirming that a manual sync completes. Escalate if the error returns.


### Verify the no-answer path


This submits an unrelated cafeteria question through the same retrieval threshold used by the supported SYNC-204 question. It asserts that no document is eligible and verifies that the answer path returns an insufficient-evidence response.

In [9]:
unsupported_question = "What is the cafeteria menu for next Tuesday?"
unsupported_documents = retrieve_knowledge(
    unsupported_question,
    max_distance=knowledge_distance_threshold,
)
unsupported_context = format_knowledge_context(unsupported_documents)
unsupported_answer = generate_answer(unsupported_question, unsupported_context)

assert unsupported_documents == []
print("Eligible evidence for unsupported question: NONE")
print(unsupported_answer)


Eligible evidence for unsupported question: NONE
I do not have enough retrieved evidence to answer that question.


## Step 5 - Prove What RAG Does Not Remember


In the first session, the user supplies two facts that are not part of the
product documentation. A stateless RAG application can record this turn in
its current prompt, but a later session only searches the document corpus.


In [10]:
TENANT_ID = "aurora-support"
USER_ID = "user:jane"
SECOND_USER_ID = "user:alex"
AGENT_ID = "agent:support-guide"
FIRST_THREAD = "thread:session-1"
NEW_THREAD = "thread:session-2"


def ensure_thread(tenant_id: str, user_id: str, agent_id: str, thread_id: str):
    sql = """
        MERGE INTO RAG_EVOLVE_THREADS target
        USING (
            SELECT :run_id run_id, :tenant_id tenant_id, :user_id user_id,
                   :agent_id agent_id, :thread_id thread_id
              FROM dual
        ) source
           ON (target.run_id = source.run_id
               AND target.tenant_id = source.tenant_id
               AND target.user_id = source.user_id
               AND target.agent_id = source.agent_id
               AND target.thread_id = source.thread_id)
        WHEN MATCHED THEN UPDATE SET target.updated_at = SYSTIMESTAMP
        WHEN NOT MATCHED THEN INSERT
            (run_id, tenant_id, user_id, agent_id, thread_id)
        VALUES
            (source.run_id, source.tenant_id, source.user_id,
             source.agent_id, source.thread_id)
    """
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                sql,
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                thread_id=thread_id,
            )
        connection.commit()


def record_turn(
    tenant_id: str,
    user_id: str,
    agent_id: str,
    thread_id: str,
    role: str,
    content: str,
) -> str:
    if role not in {"user", "assistant", "tool"}:
        raise ValueError("role must be user, assistant, or tool")
    if not content.strip():
        raise ValueError("turn content must not be empty")
    ensure_thread(tenant_id, user_id, agent_id, thread_id)
    turn_id = uuid4().hex
    sql = """
        INSERT INTO RAG_EVOLVE_TURNS
            (turn_id, run_id, tenant_id, user_id, agent_id, thread_id, role, content)
        VALUES
            (:turn_id, :run_id, :tenant_id, :user_id, :agent_id,
             :thread_id, :role, :content)
    """
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                sql,
                turn_id=turn_id,
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                thread_id=thread_id,
                role=role,
                content=content.strip(),
            )
        connection.commit()
    return turn_id


first_turn_text = (
    "I am investigating error SYNC-204. Send future support updates in Slack."
)
first_turn_id = record_turn(
    TENANT_ID, USER_ID, AGENT_ID, FIRST_THREAD, "user", first_turn_text
)
print("First-session turn recorded:", bool(first_turn_id))


First-session turn recorded: True


In [11]:
cross_session_question = "What issue was I working on, and how should you update me?"
rag_only_documents = retrieve_knowledge(cross_session_question)
rag_only_context = format_knowledge_context(rag_only_documents)

assert "user:jane" not in rag_only_context
assert "Send future support updates in Slack" not in rag_only_context
print("RAG-only context contains the user's prior turn: NO")
print(generate_answer(cross_session_question, rag_only_context))


RAG-only context contains the user's prior turn: NO
Retrieved knowledge: Never store passwords, access tokens, or authentication secrets in support notes or assistant memory. Use approved credential and secret-management systems. Before escalating a synchronization incident, collect the client version, the most recent error code, the timestamp of the failed sync, and whether a manual retry succeeded.


## Step 6 - Define the Memory Contract


A memory system should not copy every turn into a vector table. This notebook promotes only information that is durable, useful in a later session, and allowed by application policy.

The guide uses manual promotion so each write is visible and reviewable:

```text
Conversation turn
  -> candidate extraction
  -> safety and durability policy
  -> promote or reject
  -> durable scoped memory
```

A production application may use a model to propose candidates, but the model's
proposal should still pass deterministic policy checks before `remember()` runs.

| Candidate | Promote? | Reason |
|---|---:|---|
| Preferred support channel | Yes | Stable user preference |
| Current issue identifier | Yes | Useful cross-session working fact |
| Password or access token | No | Secret; use an approved system |
| Retrieved troubleshooting paragraph | No | Already belongs to the RAG corpus |
| Model reasoning | No | Transient and not user-provided evidence |

Each promoted memory has a stable `memory_key`. Writing a new active value for
the same key supersedes the old record rather than silently returning both.

This notebook serializes writes inside its Python process and adds a unique
database index that permits only one active value per scoped key. A multi-instance
production service should add retry handling around uniqueness conflicts or route
each scoped key through a single transactional writer.


## Step 7 - Add a New Agent Memory Layer


This step adds the durable memory write path. 

It defines `remember`, which embeds an approved memory, records its scope and source turn, serializes concurrent writes, and marks an older active value as superseded when the same memory key is updated. `It then uses it to promote the issue and channel preference.

In [12]:
from functools import wraps
from threading import RLock


MEMORY_WRITE_LOCK = RLock()


def serialized_memory_write(function):
    @wraps(function)
    def wrapper(*args, **kwargs):
        with MEMORY_WRITE_LOCK:
            return function(*args, **kwargs)
    return wrapper


@serialized_memory_write
def remember(
    *,
    tenant_id: str,
    user_id: str,
    agent_id: str,
    memory_key: str,
    memory_type: str,
    content: str,
    source_turn_id: str | None,
    thread_id: str | None = None,
    importance: int = 3,
) -> str:
    if memory_type not in {"preference", "fact", "episode"}:
        raise ValueError("unsupported memory_type")
    if not memory_key.strip() or not content.strip():
        raise ValueError("memory_key and content must not be empty")
    if not 1 <= int(importance) <= 5:
        raise ValueError("importance must be between 1 and 5")

    memory_id = uuid4().hex
    memory_vector = embed_texts([content.strip()])[0]
    previous_id = None
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                """
                SELECT memory_id
                  FROM RAG_EVOLVE_MEMORIES
                 WHERE run_id = :run_id
                   AND tenant_id = :tenant_id
                   AND user_id = :user_id
                   AND agent_id = :agent_id
                   AND memory_key = :memory_key
                   AND status = 'active'
                   AND ((thread_id = :thread_id) OR (thread_id IS NULL AND :thread_id IS NULL))
                 ORDER BY created_at DESC
                 FETCH FIRST 1 ROW ONLY
                """,
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                memory_key=memory_key,
                thread_id=thread_id,
            )
            row = cursor.fetchone()
            if row:
                previous_id = row[0]
                cursor.execute(
                    """
                    UPDATE RAG_EVOLVE_MEMORIES
                       SET status = 'superseded', updated_at = SYSTIMESTAMP
                     WHERE memory_id = :memory_id
                    """,
                    memory_id=previous_id,
                )

            cursor.execute(
                f"""
                INSERT INTO RAG_EVOLVE_MEMORIES
                    (memory_id, run_id, tenant_id, user_id, agent_id, thread_id,
                     memory_key, memory_type, content, embedding, status,
                     supersedes_memory_id, source_turn_id, importance)
                VALUES
                    (:memory_id, :run_id, :tenant_id, :user_id, :agent_id, :thread_id,
                     :memory_key, :memory_type, :content,
                     TO_VECTOR(:embedding_json, {VECTOR_DIM}, FLOAT32),
                     'active', :supersedes_memory_id, :source_turn_id, :importance)
                """,
                memory_id=memory_id,
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                thread_id=thread_id,
                memory_key=memory_key,
                memory_type=memory_type,
                content=content.strip(),
                embedding_json=json.dumps(memory_vector),
                supersedes_memory_id=previous_id,
                source_turn_id=source_turn_id,
                importance=int(importance),
            )
        connection.commit()
    return memory_id


In [13]:
issue_memory_id = remember(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_key="support.current_issue",
    memory_type="fact",
    content="The user is currently investigating support error SYNC-204.",
    source_turn_id=first_turn_id,
    importance=4,
)

channel_memory_id = remember(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_key="support.update_channel",
    memory_type="preference",
    content="The user prefers future support updates in Slack.",
    source_turn_id=first_turn_id,
    importance=4,
)

print("Promoted memories:", 2)


Promoted memories: 2


## Step 8 - Retrieve Memory with Explicit Scope


The following cell defines `retrieve_memories`. 

Its SQL first limits eligibility by run, tenant, user, agent, thread visibility, and active status; only then does Oracle rank the eligible rows by vector distance. This ordering prevents a similar memory from crossing an identity or conversation boundary.

In [14]:
def retrieve_memories(
    query: str,
    *,
    tenant_id: str,
    user_id: str,
    agent_id: str,
    thread_id: str,
    limit: int = 5,
) -> list[dict]:
    if not query.strip():
        raise ValueError("query must not be empty")
    limit = max(1, min(int(limit), 10))
    query_vector = embed_texts([query])[0]
    sql = f"""
        SELECT memory_id,
               memory_key,
               memory_type,
               DBMS_LOB.SUBSTR(content, 4000, 1) AS content,
               thread_id,
               importance,
               VECTOR_DISTANCE(
                   embedding,
                   TO_VECTOR(:query_vector, {VECTOR_DIM}, FLOAT32),
                   COSINE
               ) AS distance
          FROM RAG_EVOLVE_MEMORIES
         WHERE run_id = :run_id
           AND tenant_id = :tenant_id
           AND user_id = :user_id
           AND agent_id = :agent_id
           AND status = 'active'
           AND (thread_id IS NULL OR thread_id = :thread_id)
         ORDER BY importance DESC, distance
         FETCH FIRST {limit} ROWS ONLY
    """
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            rows = cursor.execute(
                sql,
                query_vector=json.dumps(query_vector),
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                thread_id=thread_id,
            ).fetchall()
    return [
        {
            "memory_id": row[0],
            "memory_key": row[1],
            "memory_type": row[2],
            "content": row[3],
            "thread_id": row[4],
            "importance": int(row[5]),
            "distance": float(row[6]),
        }
        for row in rows
    ]


recalled_memories = retrieve_memories(
    cross_session_question,
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=NEW_THREAD,
)
pd.DataFrame(recalled_memories)[
    ["memory_key", "memory_type", "importance", "distance"]
]


,memory_key,memory_type,importance,distance
0,support.update_channel,preference,4,0.959839
1,support.current_issue,fact,4,0.996913


### Prove that scope is an eligibility boundary


Now, we repeat the question under a second user identity. 

The empty result demonstrates that Jane's memories are excluded before similarity ranking, rather than merely hidden after retrieval.

In [15]:
isolated_user_results = retrieve_memories(
    cross_session_question,
    tenant_id=TENANT_ID,
    user_id=SECOND_USER_ID,
    agent_id=AGENT_ID,
    thread_id="thread:alex-1",
)

assert isolated_user_results == []
print("Second user received Jane's memories: NO")


Second user received Jane's memories: NO


### Validate tenant, agent, and thread boundaries


This creates one thread-scoped episode and queries it from matching and nonmatching scopes. 

Its assertions cover tenant isolation, agent isolation, thread isolation, and the continued visibility of user-level memories across threads.

In [16]:
thread_memory_id = remember(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=FIRST_THREAD,
    memory_key="thread.follow_up",
    memory_type="episode",
    content="In this thread, follow up after the user retries the sync.",
    source_turn_id=first_turn_id,
    importance=3,
)

same_thread_results = retrieve_memories(
    "What follow-up is needed?",
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=FIRST_THREAD,
)
other_thread_results = retrieve_memories(
    "What follow-up is needed?",
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=NEW_THREAD,
)
other_tenant_results = retrieve_memories(
    cross_session_question,
    tenant_id="another-tenant",
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id=NEW_THREAD,
)
other_agent_results = retrieve_memories(
    cross_session_question,
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id="agent:other",
    thread_id=NEW_THREAD,
)

assert any(item["memory_key"] == "thread.follow_up" for item in same_thread_results)
assert all(item["memory_key"] != "thread.follow_up" for item in other_thread_results)
assert other_tenant_results == []
assert other_agent_results == []
print("Tenant, agent, and thread scope checks: PASS")


Tenant, agent, and thread scope checks: PASS


## Step 9 - Simulate an Application Restart


A notebook cannot restart its own kernel and continue automatically, but it can test the same persistence boundary. 

This step closes the entire database pool, removes the previously retrieved Python objects, creates a new pool, and reconstructs memory using only durable identifiers and Oracle data.

In [17]:
pre_restart_memory_keys = {item["memory_key"] for item in recalled_memories}
pool.close(force=True)
del recalled_memories
del first_turn_text

pool = oracledb.create_pool(
    user=CONFIG["DB_USER"],
    password=CONFIG["DB_PASSWORD"],
    dsn=CONFIG["DB_DSN"],
    min=1,
    max=4,
    increment=1,
)

resumed_memories = retrieve_memories(
    cross_session_question,
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id="thread:after-restart",
)
resumed_memory_keys = {item["memory_key"] for item in resumed_memories}

assert {"support.current_issue", "support.update_channel"}.issubset(
    resumed_memory_keys
)
assert pre_restart_memory_keys.issubset(resumed_memory_keys)
print("New database pool: READY")
print("Durable memories recovered without prior retrieval state: PASS")


New database pool: READY
Durable memories recovered without prior retrieval state: PASS


## Step 10 - Assemble RAG and Memory Context


The following cell retrieves documents and memories independently, then combines them through `assemble_context`. It keeps durable user memory and knowledge-base evidence in separately labeled sections so the answer step can preserve their different provenance instead of blending them into an unlabeled prompt.

In [18]:
def assemble_context(documents: list[dict], memories: list[dict]) -> str:
    memory_section = "\n\n".join(
        f"[MEMORY {item['memory_type']}:{item['memory_key']}]\n{item['content']}"
        for item in memories
    ) or "[No eligible durable memories]"
    document_section = format_knowledge_context(documents) or "[No matching documents]"
    return (
        "DURABLE USER MEMORY\n"
        + memory_section
        + "\n\nRETRIEVED KNOWLEDGE\n"
        + document_section
    )


memory_aware_documents = retrieve_knowledge(cross_session_question)
memory_aware_context = assemble_context(
    memory_aware_documents, resumed_memories
)
print(memory_aware_context)


DURABLE USER MEMORY
[MEMORY preference:support.update_channel]
The user prefers future support updates in Slack.

[MEMORY fact:support.current_issue]
The user is currently investigating support error SYNC-204.

RETRIEVED KNOWLEDGE
[DOCUMENT account-security.md#account-security]
Never store passwords, access tokens, or authentication secrets in support notes or assistant memory. Use approved credential and secret-management systems.

[DOCUMENT aurora-sync-diagnostics.md#sync-diagnostics]
Before escalating a synchronization incident, collect the client version, the most recent error code, the timestamp of the failed sync, and whether a manual retry succeeded.

[DOCUMENT aurora-sync-troubleshooting.md#sync-204-resolution]
Error SYNC-204 means the desktop client lost its refresh token. Resolve it by signing out, removing the cached session, signing in again, and confirming that a manual sync completes. Escalate if the error returns.


In [19]:
memory_aware_answer = generate_answer(
    cross_session_question, memory_aware_context
)
print(memory_aware_answer)


Remembered context: The user prefers future support updates in Slack. The user is currently investigating support error SYNC-204.

Retrieved knowledge: Never store passwords, access tokens, or authentication secrets in support notes or assistant memory. Use approved credential and secret-management systems. Before escalating a synchronization incident, collect the client version, the most recent error code, the timestamp of the failed sync, and whether a manual retry succeeded.


## Step 11 - Close the Turn Loop


The application loop now has two retrieval paths and one guarded write path:

1. identify the tenant, user, agent, and thread;
2. retrieve active memories within that scope;
3. retrieve relevant knowledge chunks;
4. assemble bounded context with labeled sources;
5. generate and record the response;
6. promote only durable candidates allowed by policy.

Retrieval does not automatically grant permission, and generation does not
automatically create memory. Those remain explicit application decisions.


In [20]:
assistant_turn_id = record_turn(
    TENANT_ID,
    USER_ID,
    AGENT_ID,
    NEW_THREAD,
    "assistant",
    memory_aware_answer,
)
print("Memory-aware assistant turn recorded:", bool(assistant_turn_id))


Memory-aware assistant turn recorded: True


## Step 12 - Reject Unsafe Memory Candidates


A useful memory system needs a rejection path. The policy below blocks common
credential-like values before they reach durable turn or memory storage. The
example uses a synthetic placeholder; never put a real secret in a notebook.


In [21]:
SENSITIVE_MEMORY_PATTERNS = [
    re.compile(r"\bpassword\s*(?:is|:|=)", re.IGNORECASE),
    re.compile(r"\bapi[_ -]?key\s*(?:is|:|=)", re.IGNORECASE),
    re.compile(r"\baccess[_ -]?token\s*(?:is|:|=)", re.IGNORECASE),
]


def contains_sensitive_candidate(text: str) -> bool:
    return any(pattern.search(text) for pattern in SENSITIVE_MEMORY_PATTERNS)


def promote_memory_candidate(
    *,
    tenant_id: str,
    user_id: str,
    agent_id: str,
    memory_key: str,
    memory_type: str,
    content: str,
    source_turn_id: str | None,
    thread_id: str | None = None,
    importance: int = 3,
) -> dict:
    if contains_sensitive_candidate(content):
        return {
            "status": "rejected",
            "reason": "credential-like content is not eligible for durable memory",
            "memory_id": None,
        }
    memory_id = remember(
        tenant_id=tenant_id,
        user_id=user_id,
        agent_id=agent_id,
        memory_key=memory_key,
        memory_type=memory_type,
        content=content,
        source_turn_id=source_turn_id,
        thread_id=thread_id,
        importance=importance,
    )
    return {"status": "promoted", "reason": "policy passed", "memory_id": memory_id}


unsafe_candidate = "My password is tutorial-secret-not-a-real-credential."
sanitized_turn_id = record_turn(
    TENANT_ID,
    USER_ID,
    AGENT_ID,
    "thread:safety-check",
    "user",
    "The user supplied credential-like content; the value was redacted.",
)
unsafe_promotion = promote_memory_candidate(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_key="unsafe.credential",
    memory_type="fact",
    content=unsafe_candidate,
    source_turn_id=sanitized_turn_id,
    importance=5,
)

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        stored_secret_count = cursor.execute(
            """
            SELECT
                (SELECT COUNT(*) FROM RAG_EVOLVE_TURNS
                  WHERE run_id = :run_id
                    AND DBMS_LOB.INSTR(content, :needle) > 0)
              + (SELECT COUNT(*) FROM RAG_EVOLVE_MEMORIES
                  WHERE run_id = :run_id
                    AND DBMS_LOB.INSTR(content, :needle) > 0)
            FROM dual
            """,
            run_id=RUN_ID,
            needle="tutorial-secret",
        ).fetchone()[0]

assert unsafe_promotion["status"] == "rejected"
assert stored_secret_count == 0
print("Unsafe candidate promoted: NO")
print("Synthetic credential value stored: NO")


Unsafe candidate promoted: NO
Synthetic credential value stored: NO


## Step 13 - Correct and Forget Memories


Durable memory needs a lifecycle, not just insertion and retrieval. The next two examples first update a preference by creating a new active version and superseding the old one, then mark an issue memory as deleted so it becomes ineligible while its audit history remains in Oracle.

### Correct a preference without erasing its history


The following cell records the user's change from Slack to email and calls `remember` with the existing memory key. It then verifies the new active value, the superseded row, and the link between the two versions.

In [22]:
correction_turn_id = record_turn(
    TENANT_ID,
    USER_ID,
    AGENT_ID,
    "thread:session-3",
    "user",
    "Use email for future support updates, not Slack.",
)

corrected_channel_id = remember(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_key="support.update_channel",
    memory_type="preference",
    content="The user prefers future support updates by email.",
    source_turn_id=correction_turn_id,
    importance=4,
)

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        channel_history = cursor.execute(
            """
            SELECT status, DBMS_LOB.SUBSTR(content, 4000, 1), supersedes_memory_id
              FROM RAG_EVOLVE_MEMORIES
             WHERE run_id = :run_id
               AND user_id = :user_id
               AND memory_key = 'support.update_channel'
             ORDER BY created_at
            """,
            run_id=RUN_ID,
            user_id=USER_ID,
        ).fetchall()

assert [row[0] for row in channel_history] == ["superseded", "active"]
assert "email" in channel_history[-1][1].lower()
assert channel_history[-1][2] == channel_memory_id
corrected_memories = retrieve_memories(
    "How should you send future support updates?",
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id="thread:session-4",
)
corrected_context = assemble_context([], corrected_memories)
assert "email" in corrected_context.lower()
assert "Slack" not in corrected_context
pd.DataFrame(
    channel_history,
    columns=["status", "content", "supersedes_memory_id"],
)


,status,content,supersedes_memory_id
0,superseded,The user prefers future support updates in Slack.,NaN
1,active,The user prefers future support updates by email.,3abda14e9b8b4da88373c5824e1ec7ab


### Forget a memory by making it ineligible for retrieval


The following cell defines `forget_memory` as a scoped status update rather than a hard delete. After marking the current issue as deleted, it runs retrieval again and asserts that the forgotten memory is no longer eligible for context assembly.

In [23]:
def forget_memory(
    *,
    tenant_id: str,
    user_id: str,
    agent_id: str,
    memory_key: str,
    thread_id: str | None = None,
) -> int:
    with pool.acquire() as connection:
        with connection.cursor() as cursor:
            cursor.execute(
                """
                UPDATE RAG_EVOLVE_MEMORIES
                   SET status = 'deleted', updated_at = SYSTIMESTAMP
                 WHERE run_id = :run_id
                   AND tenant_id = :tenant_id
                   AND user_id = :user_id
                   AND agent_id = :agent_id
                   AND memory_key = :memory_key
                   AND ((thread_id = :thread_id) OR (thread_id IS NULL AND :thread_id IS NULL))
                   AND status = 'active'
                """,
                run_id=RUN_ID,
                tenant_id=tenant_id,
                user_id=user_id,
                agent_id=agent_id,
                memory_key=memory_key,
                thread_id=thread_id,
            )
            deleted_count = cursor.rowcount
        connection.commit()
    return deleted_count


forgotten_count = forget_memory(
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    memory_key="support.current_issue",
    thread_id=None,
)
after_forget = retrieve_memories(
    "What issue was I working on?",
    tenant_id=TENANT_ID,
    user_id=USER_ID,
    agent_id=AGENT_ID,
    thread_id="thread:session-5",
)

assert forgotten_count == 1
assert all(item["memory_key"] != "support.current_issue" for item in after_forget)
print("Forgotten issue memory returned by retrieval: NO")


Forgotten issue memory returned by retrieval: NO


## Step 14 - Evaluate the Evolution


This step gathers the notebook's assertions into a single evaluation table. It reports retrieval, abstention, cross-session persistence, identity isolation, correction, secret rejection, and forgetting, while marking provider-specific checks as `NOT RUN` when they were not executed.

In [24]:
evaluation = pd.DataFrame(
    [
        {
            "check": "RAG retrieves the SYNC-204 procedure",
            "result": baseline_documents[0]["chunk_id"] == "sync-204-resolution",
        },
        {
            "check": "Semantic provider retrieves the paraphrased problem when enabled",
            "result": (
                semantic_paraphrase_pass
                if CONFIG["EMBEDDING_PROVIDER"] == "openai"
                else None
            ),
        },
        {
            "check": "Unsupported question produces no eligible evidence",
            "result": unsupported_documents == [],
        },
        {
            "check": "RAG-only context excludes the prior user turn",
            "result": "Send future support updates in Slack" not in rag_only_context,
        },
        {
            "check": "Memory recalls the issue across threads",
            "result": any(
                item["memory_key"] == "support.current_issue"
                for item in resumed_memories
            ),
        },
        {
            "check": "Memory survives a closed and recreated database pool",
            "result": {"support.current_issue", "support.update_channel"}.issubset(
                resumed_memory_keys
            ),
        },
        {
            "check": "Memory recalls the original Slack preference",
            "result": "Slack" in memory_aware_context,
        },
        {
            "check": "A second user cannot retrieve Jane's memories",
            "result": isolated_user_results == [],
        },
        {
            "check": "Tenant scope excludes the same user in another tenant",
            "result": other_tenant_results == [],
        },
        {
            "check": "Agent scope excludes another agent",
            "result": other_agent_results == [],
        },
        {
            "check": "Thread memory stays in its originating thread",
            "result": any(
                item["memory_key"] == "thread.follow_up"
                for item in same_thread_results
            ) and all(
                item["memory_key"] != "thread.follow_up"
                for item in other_thread_results
            ),
        },
        {
            "check": "Correction returns email and excludes Slack",
            "result": "email" in corrected_context.lower()
            and "Slack" not in corrected_context,
        },
        {
            "check": "Credential-like candidate is rejected and never stored",
            "result": unsafe_promotion["status"] == "rejected"
            and stored_secret_count == 0,
        },
        {
            "check": "Forgotten issue memory is no longer eligible",
            "result": all(
                item["memory_key"] != "support.current_issue"
                for item in after_forget
            ),
        },
    ]
)

executed_results = evaluation["result"].dropna()
assert executed_results.all(), evaluation
evaluation.assign(
    result=evaluation["result"].map(
        lambda value: (
            "NOT RUN"
            if pd.isna(value)
            else ("PASS" if bool(value) else "FAIL")
        )
    )
)


,check,result
0,RAG retrieves the SYNC-204 procedure,PASS
1,Semantic provider retrieves the paraphrased pr...,NOT RUN
2,Unsupported question produces no eligible evid...,PASS
3,RAG-only context excludes the prior user turn,PASS
4,Memory recalls the issue across threads,PASS
5,Memory survives a closed and recreated databas...,PASS
6,Memory recalls the original Slack preference,PASS
7,A second user cannot retrieve Jane's memories,PASS
8,Tenant scope excludes the same user in another...,PASS
9,Agent scope excludes another agent,PASS


## Step 15 - Inspect the Oracle State


The final inspection cell counts the knowledge chunks, threads, turns, and each memory lifecycle state created under this run's `run_id`. This makes the effect of the tutorial visible in Oracle before the remaining cells perform optional cleanup and close the connection pool.

In [25]:
inspection_queries = {
    "knowledge_chunks": "SELECT COUNT(*) FROM RAG_EVOLVE_KNOWLEDGE WHERE run_id = :run_id",
    "threads": "SELECT COUNT(*) FROM RAG_EVOLVE_THREADS WHERE run_id = :run_id",
    "turns": "SELECT COUNT(*) FROM RAG_EVOLVE_TURNS WHERE run_id = :run_id",
    "active_memories": "SELECT COUNT(*) FROM RAG_EVOLVE_MEMORIES WHERE run_id = :run_id AND status = 'active'",
    "superseded_memories": "SELECT COUNT(*) FROM RAG_EVOLVE_MEMORIES WHERE run_id = :run_id AND status = 'superseded'",
    "deleted_memories": "SELECT COUNT(*) FROM RAG_EVOLVE_MEMORIES WHERE run_id = :run_id AND status = 'deleted'",
}

with pool.acquire() as connection:
    with connection.cursor() as cursor:
        inspection = {
            label: cursor.execute(sql, run_id=RUN_ID).fetchone()[0]
            for label, sql in inspection_queries.items()
        }

pd.DataFrame(
    [{"object": label, "rows": count} for label, count in inspection.items()]
)


,object,rows
0,knowledge_chunks,4
1,threads,4
2,turns,4
3,active_memories,2
4,superseded_memories,1
5,deleted_memories,1


## What Changed from RAG to Agent Memory?

The original RAG application already knew how to retrieve external evidence.
The evolution added five responsibilities:

1. **Identity:** tenant, user, agent, and thread are explicit inputs.
2. **Capture:** turns are stored as interaction evidence.
3. **Promotion:** only selected durable information becomes memory.
4. **Lifecycle:** active, superseded, and deleted memories remain distinguishable.
5. **Context assembly:** memory and documents are retrieved independently and labeled before generation.

This is why an agent memory system is more than a second vector collection.
It is a state-management and context-selection system built around interaction.


## Optional Cleanup


In [26]:
CLEANUP_RUN = os.getenv("CLEANUP_RAG_MEMORY_RUN", "0") == "1"

if CLEANUP_RUN:
    with oracledb.connect(
        user=CONFIG["DB_USER"],
        password=CONFIG["DB_PASSWORD"],
        dsn=CONFIG["DB_DSN"],
    ) as connection:
        with connection.cursor() as cursor:
            for table_name in [
                "RAG_EVOLVE_MEMORIES",
                "RAG_EVOLVE_TURNS",
                "RAG_EVOLVE_THREADS",
                "RAG_EVOLVE_KNOWLEDGE",
            ]:
                cursor.execute(
                    f"DELETE FROM {table_name} WHERE run_id = :run_id",
                    run_id=RUN_ID,
                )
        connection.commit()
    print("This notebook run's rows were removed.")
else:
    print("Cleanup skipped. Set CLEANUP_RAG_MEMORY_RUN=1 and rerun this cell to remove this run's rows.")


Cleanup skipped. Set CLEANUP_RAG_MEMORY_RUN=1 and rerun this cell to remove this run's rows.


In [27]:
try:
    pool.close(force=True)
    print("Oracle connection pool closed.")
except oracledb.Error:
    print("Oracle connection pool was already closed.")


Oracle connection pool closed.


## Summary

This notebook covered the complete evolution from a stateless RAG application to a durable agent memory system:

- creating and querying an Oracle vector-backed knowledge base
- showing why retrieved documents do not replace interaction memory
- introducing tenant, user, agent, and thread identity
- promoting useful facts and preferences through an explicit policy boundary
- recovering memory after database sessions are closed and recreated
- assembling labeled RAG evidence and scoped memories for generation
- rejecting credential-like memory candidates
- correcting, superseding, and forgetting durable memories
- validating no-answer behavior, isolation boundaries, and lifecycle outcomes


## Production Considerations

- Apply authorization before similarity ranking; a high score never grants access.
- Define which facts may be promoted, their retention periods, and who can delete them.
- Keep authoritative business data outside memory and retrieve it through approved tools.
- Add conflict detection, freshness checks, and human review for sensitive memories.
- Evaluate exact identifiers, paraphrases, corrections, user isolation, and no-answer cases.
- Use vector indexes and bounded context budgets when the corpus grows.
- Record why a memory was selected so the assembled context remains inspectable.


## References

- [Oracle AI Vector Search documentation](https://docs.oracle.com/en/database/oracle/oracle-database/26/vecse/)
- [python-oracledb documentation](https://python-oracledb.readthedocs.io/)
- [OpenAI Responses API text-generation guide](https://developers.openai.com/api/docs/guides/text)
- [OpenAI embeddings guide](https://developers.openai.com/api/docs/guides/embeddings)
